# Inference Playground

Backward-compatible notebook for evaluation and qualitative visualization.


In [ ]:
from pathlib import Path

REPO_DIR = Path(globals().get('REPO_DIR', '/content/Military_Object_Detection'))
WORKSPACE_ROOT = Path(globals().get('WORKSPACE_ROOT', '/content/drive/MyDrive/Military_Object_Detection'))
RESULTS_ROOT = Path(globals().get('RESULTS_ROOT', WORKSPACE_ROOT / 'experiments' / 'colab_quick'))
DATASET_YAML = Path(globals().get('DATASET_YAML', WORKSPACE_ROOT / 'data' / 'processed' / 'yolo_dataset' / 'dataset.yaml'))
MODEL_PATH = globals().get('MODEL_PATH', None)
SPLIT = globals().get('SPLIT', 'test')
CONF = float(globals().get('CONF', 0.25))
N_SAMPLES = int(globals().get('N_SAMPLES', 8))
NOTEBOOK_OUTPUT_DIR = Path(globals().get('NOTEBOOK_OUTPUT_DIR', WORKSPACE_ROOT / 'artifacts' / 'notebook_eval'))

print('RESULTS_ROOT       :', RESULTS_ROOT)
print('DATASET_YAML       :', DATASET_YAML)
print('NOTEBOOK_OUTPUT_DIR:', NOTEBOOK_OUTPUT_DIR)


In [ ]:
import os
import sys
from pathlib import Path

REPO_DIR = Path(globals().get('REPO_DIR', '/content/Military_Object_Detection'))
WORKSPACE_ROOT = Path(globals().get('WORKSPACE_ROOT', '/content/drive/MyDrive/Military_Object_Detection'))
if not REPO_DIR.exists():
    raise FileNotFoundError(f'Repository not found at {REPO_DIR}. Run notebooks/00_colab_setup.ipynb first.')

os.environ['MAD_WORKSPACE_ROOT'] = str(WORKSPACE_ROOT)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

import cv2
import matplotlib.pyplot as plt
import pandas as pd
import yaml
from ultralytics import YOLO

from mad.inference import evaluate_model


In [ ]:
from pathlib import Path
import pandas as pd

results_root = Path(RESULTS_ROOT)
runs_csv = results_root / 'benchmark_all_runs.csv'
if not runs_csv.exists():
    study_candidates = sorted(results_root.glob('study_*/benchmark_all_runs.csv'))
    if study_candidates:
        runs_csv = study_candidates[-1]

if MODEL_PATH is None:
    if not runs_csv.exists():
        raise FileNotFoundError(f'Could not find benchmark results under {results_root}')
    df = pd.read_csv(runs_csv)
    if 'status' in df.columns:
        df = df[df['status'] == 'ok'].copy()
    rank_col = 'test_map50_95' if 'test_map50_95' in df.columns else 'val_map50_95'
    if df.empty:
        raise RuntimeError('No successful benchmark runs were found.')
    MODEL_PATH = Path(df.sort_values(rank_col, ascending=False).iloc[0]['best_model'])
else:
    MODEL_PATH = Path(MODEL_PATH)

print('Benchmark CSV:', runs_csv)
print('Selected model:', MODEL_PATH)


In [ ]:
from pathlib import Path

eval_output_dir = Path(NOTEBOOK_OUTPUT_DIR) / 'evaluation'
metrics = evaluate_model(
    model_path=Path(MODEL_PATH),
    dataset_yaml=Path(DATASET_YAML),
    split=SPLIT,
    device='auto',
    output_dir=eval_output_dir,
)
metrics


In [ ]:
import random
from pathlib import Path

dataset_cfg = yaml.safe_load(Path(DATASET_YAML).read_text(encoding='utf-8'))
dataset_root = Path(dataset_cfg.get('path', Path(DATASET_YAML).parent))
if not dataset_root.is_absolute():
    dataset_root = (Path(DATASET_YAML).parent / dataset_root).resolve()

split_path = Path(dataset_cfg.get(SPLIT, dataset_cfg.get('val')))
if not split_path.is_absolute():
    split_path = (dataset_root / split_path).resolve()
if split_path.suffix.lower() == '.txt':
    image_paths = [Path(line.strip()) for line in split_path.read_text(encoding='utf-8').splitlines() if line.strip()]
else:
    image_paths = [p for p in sorted(split_path.glob('*')) if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}]

if not image_paths:
    raise RuntimeError(f'No images were found for split {SPLIT}: {split_path}')

sample_paths = random.sample(image_paths, k=min(N_SAMPLES, len(image_paths)))
model = YOLO(str(MODEL_PATH))
results = model.predict([str(path) for path in sample_paths], conf=CONF, verbose=False)


In [ ]:
ncols = 2
nrows = (len(results) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows))
axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]

for ax, result in zip(axes, results):
    plotted = cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB)
    ax.imshow(plotted)
    ax.set_title(Path(result.path).name)
    ax.axis('off')

for ax in axes[len(results):]:
    ax.axis('off')

plt.tight_layout()
plt.show()
print('Evaluation artifacts:', eval_output_dir)
